# Base CNN Diffusion Policy Training (Normalized)

이 노트북은 `base_cnn_train.ipynb`의 정규화 버전입니다.

- `observations`, `action_chunks_t`를 평균 0 / 표준편차 1로 정규화합니다.
- 학습은 정규화된 action chunk 기준으로 수행합니다.
- 샘플링 결과는 다시 원래 action 스케일로 역정규화해서 확인합니다.
- `pred_noise`의 평균과 표준편차를 함께 출력해서 `pred_noise ~= 0` 붕괴 여부를 확인합니다.


In [1]:
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from based_cnn_model import DiffusionPolicy


/home/kdy1210/ai_architecture_study/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass
class TrainConfig:
    seed: int = 42
    data_path: str = "./datas/reach_bc_imitation_h15.npz"
    horizon: int = 15
    diffusion_steps: int = 100
    obs_latent: int = 128
    pos_latent: int = 64
    batch_size: int = 64
    num_epochs: int = 20
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    beta_start: float = 1e-4
    beta_end: float = 5e-2
    log_every: int = 200
    eps: float = 1e-6

config = TrainConfig()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)

print(device)


cuda:1


In [3]:
data = np.load(config.data_path)
observations = data["observations"].astype(np.float32)
action_chunks = data["action_chunks"].astype(np.float32)
action_chunks_t = data["action_chunks_t"].astype(np.float32)

config.horizon = int(data["horizon"])
obs_dim = int(data["obs_dim"])
action_dim = int(data["action_dim"])

print(f"dataset={config.data_path}")
print(f"observations={observations.shape}")
print(f"action_chunks={action_chunks.shape}")
print(f"action_chunks_t={action_chunks_t.shape}")
print(f"obs_dim={obs_dim}, action_dim={action_dim}, horizon={config.horizon}")


dataset=./datas/reach_bc_imitation_h15.npz
observations=(1000000, 32)
action_chunks=(1000000, 15, 7)
action_chunks_t=(1000000, 7, 15)
obs_dim=32, action_dim=7, horizon=15


In [4]:
obs_mean = observations.mean(axis=0, keepdims=True)
obs_std = observations.std(axis=0, keepdims=True)
obs_std = np.clip(obs_std, config.eps, None)

act_mean = action_chunks_t.mean(axis=(0, 2), keepdims=True)
act_std = action_chunks_t.std(axis=(0, 2), keepdims=True)
act_std = np.clip(act_std, config.eps, None)

observations_norm = ((observations - obs_mean) / obs_std).astype(np.float32)
action_chunks_t_norm = ((action_chunks_t - act_mean) / act_std).astype(np.float32)

def summarize(name: str, array: np.ndarray):
    print(
        f"{name}: shape={array.shape}, mean={array.mean():.6f}, std={array.std():.6f}, "
        f"min={array.min():.6f}, max={array.max():.6f}"
    )

summarize("observations(raw)", observations)
summarize("observations(norm)", observations_norm)
summarize("action_chunks_t(raw)", action_chunks_t)
summarize("action_chunks_t(norm)", action_chunks_t_norm)


observations(raw): shape=(1000000, 32), mean=-4.129064, std=49.487072, min=-556.328430, max=245.340820
observations(norm): shape=(1000000, 32), mean=-0.001177, std=0.969504, min=-403.122528, max=205.093185
action_chunks_t(raw): shape=(1000000, 7, 15), mean=-19.661648, std=104.986488, min=-556.328430, max=245.340820
action_chunks_t(norm): shape=(1000000, 7, 15), mean=-0.000041, std=1.000285, min=-5.852469, max=9.202013


In [5]:
class ReachDiffusionDataset(Dataset):
    def __init__(self, observations_norm: np.ndarray, action_chunks_t_norm: np.ndarray):
        self.observations = torch.from_numpy(observations_norm)
        self.action_chunks_t = torch.from_numpy(action_chunks_t_norm)

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        return {
            "obs": self.observations[idx],
            "action_chunk": self.action_chunks_t[idx],
        }


In [6]:
dataset = ReachDiffusionDataset(observations_norm, action_chunks_t_norm)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, drop_last=True)

batch = next(iter(dataloader))
print(batch["obs"].shape, batch["action_chunk"].shape)
print(f"dataset size={len(dataset):,}")
print(f"steps per epoch={len(dataloader):,}")


torch.Size([64, 32]) torch.Size([64, 7, 15])
dataset size=1,000,000
steps per epoch=15,625


In [7]:
model = DiffusionPolicy(
    action_dim=action_dim,
    obs_dim=obs_dim,
    obs_latent=config.obs_latent,
    pos_latent=config.pos_latent,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

betas, alphas, alpha_bars = model.get_b_a_ab(
    start=config.beta_start,
    end=config.beta_end,
    timestep=config.diffusion_steps,
)
alpha_bars = alpha_bars.to(device)

print(f"num_params={sum(p.numel() for p in model.parameters()):,}")
print(f"alpha_bar_last={alpha_bars[-1].item():.6f}")


num_params=207,367
alpha_bar_last=0.078234


In [8]:
def q_sample_actions(x0: torch.Tensor, t_index: torch.Tensor, alpha_bars: torch.Tensor):
    noise = torch.randn_like(x0)
    alpha_bar_t = alpha_bars[t_index].view(-1, 1, 1)
    xt = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1.0 - alpha_bar_t) * noise
    return xt, noise


def train_one_epoch(model, dataloader, optimizer, alpha_bars, diffusion_steps, device, log_every, epoch):
    model.train()
    running_loss = 0.0
    running_pred_mean = 0.0
    running_pred_std = 0.0
    progress = tqdm(dataloader, desc=f"epoch {epoch:03d}", leave=False)

    for step, batch in enumerate(progress, start=1):
        obs = batch["obs"].to(device)
        action_chunk = batch["action_chunk"].to(device)

        t_index = torch.randint(0, diffusion_steps, (obs.shape[0],), device=device)
        noisy_action, noise = q_sample_actions(action_chunk, t_index, alpha_bars)

        pred_noise = model(noisy_action, t_index.float(), obs)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        running_pred_mean += pred_noise.detach().mean().item()
        running_pred_std += pred_noise.detach().std().item()

        if step % log_every == 0 or step == 1:
            progress.set_postfix(
                loss=f"{running_loss / step:.6f}",
                pred_mean=f"{running_pred_mean / step:.4f}",
                pred_std=f"{running_pred_std / step:.4f}",
            )

    num_steps = max(len(dataloader), 1)
    return {
        "loss": running_loss / num_steps,
        "pred_mean": running_pred_mean / num_steps,
        "pred_std": running_pred_std / num_steps,
    }


In [9]:
history = []

for epoch in tqdm(range(1, config.num_epochs + 1), desc="training epochs"):
    metrics = train_one_epoch(
        model=model,
        dataloader=dataloader,
        optimizer=optimizer,
        alpha_bars=alpha_bars,
        diffusion_steps=config.diffusion_steps,
        device=device,
        log_every=config.log_every,
        epoch=epoch,
    )
    history.append(metrics)
    print(
        f"epoch={epoch:03d} "
        f"mean_loss={metrics['loss']:.6f} "
        f"pred_mean={metrics['pred_mean']:.4f} "
        f"pred_std={metrics['pred_std']:.4f}"
    )


training epochs:   5%|▌         | 1/20 [01:19<25:06, 79.27s/it]

epoch=001 mean_loss=0.117366 pred_mean=-0.0000 pred_std=0.9343


training epochs:  10%|█         | 2/20 [02:42<24:27, 81.52s/it]

epoch=002 mean_loss=0.061686 pred_mean=0.0001 pred_std=0.9682


training epochs:  15%|█▌        | 3/20 [03:59<22:30, 79.46s/it]

...

epoch=018 mean_loss=0.029301 pred_mean=-0.0001 pred_std=0.9854


training epochs:  95%|█████████▌| 19/20 [25:35<01:18, 78.20s/it]

epoch=019 mean_loss=0.029010 pred_mean=-0.0001 pred_std=0.9855


training epochs: 100%|██████████| 20/20 [26:56<00:00, 80.81s/it]

epoch=020 mean_loss=0.028741 pred_mean=0.0001 pred_std=0.9856


In [10]:
@torch.no_grad()
def sample_action_chunk(model, obs: torch.Tensor, horizon: int, action_dim: int, alpha_bars: torch.Tensor, beta_start: float, beta_end: float, diffusion_steps: int):
    model.eval()
    x = torch.randn(1, action_dim, horizon, device=obs.device)

    betas, alphas, _ = model.get_b_a_ab(beta_start, beta_end, diffusion_steps)
    betas = betas.to(obs.device)
    alphas = alphas.to(obs.device)

    for step in reversed(range(diffusion_steps)):
        t_batch = torch.full((1,), float(step), device=obs.device)
        pred_noise = model(x, t_batch, obs)

        alpha_t = alphas[step]
        alpha_bar_t = alpha_bars[step]
        coef = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)
        x = (x - coef * pred_noise) / torch.sqrt(alpha_t)

        if step > 0:
            x = x + torch.sqrt(betas[step]) * torch.randn_like(x)

    return x.squeeze(0).transpose(0, 1)


def denormalize_action_chunk(action_chunk_norm: torch.Tensor, act_mean: np.ndarray, act_std: np.ndarray) -> torch.Tensor:
    act_mean_t = torch.from_numpy(act_mean.astype(np.float32)).to(action_chunk_norm.device).view(1, action_chunk_norm.shape[-1])
    act_std_t = torch.from_numpy(act_std.astype(np.float32)).to(action_chunk_norm.device).view(1, action_chunk_norm.shape[-1])
    return action_chunk_norm * act_std_t + act_mean_t


In [11]:
test_index = 0

test_obs = torch.from_numpy(observations_norm[test_index]).unsqueeze(0).to(device)
gt_chunk_norm = torch.from_numpy(action_chunks[test_index]).to(device)

sampled_chunk_norm = sample_action_chunk(
    model=model,
    obs=test_obs,
    horizon=config.horizon,
    action_dim=action_dim,
    alpha_bars=alpha_bars,
    beta_start=config.beta_start,
    beta_end=config.beta_end,
    diffusion_steps=config.diffusion_steps,
)

sampled_chunk = denormalize_action_chunk(sampled_chunk_norm, act_mean.squeeze(0), act_std.squeeze(0))

print("ground truth chunk[0]")
print(gt_chunk_norm.detach().cpu())
print()
print("sampled chunk[0] (denormalized)")
print(sampled_chunk.detach().cpu())


ground truth chunk[0]
tensor([[  17.6552,   12.1108,   28.4821,   -9.6592,    0.3618,   24.7958,
           -3.3527],
        [  37.1561,   33.6210,   52.4586,  -22.2621,    6.3087,   47.0246,
          -12.6862],
        [  54.6105,   46.3145,   73.1615,  -36.7046,    8.3309,   65.2239,
          -22.6299],
        [  63.4201,   50.3302,   82.7867,  -48.4775,    5.0422,   80.4293,
          -33.8365],
        [  65.1505,   52.5929,   88.9083,  -56.5057,    1.6086,   92.4904,
          -40.7903],
        [  63.1479,   53.4615,   93.5099,  -63.7312,   -1.2676,  101.8695,
          -44.4581],
        [  59.6954,   51.6785,   97.7796,  -71.8862,   -3.9469,  109.6363,
          -47.3215],
        [  55.6391,   49.2588,  102.1410,  -80.9340,   -6.3348,  116.1670,
          -49.5922],
        [  51.6252,   46.8609,  106.6469,  -90.3805,   -8.4612,  121.8657,
          -51.6328],
        [  48.3217,   45.1590,  110.5054, -100.1055,  -10.3015,  127.6983,
          -53.9696],
        [  44.4640

In [12]:
loss_history = [item["loss"] for item in history]
pred_std_history = [item["pred_std"] for item in history]

print("loss history")
print(loss_history)
print()
print("pred std history")
print(pred_std_history)


loss history
[0.11736559439492225, 0.06168597669196129, 0.04958774098449945, 0.04405483297300339, 0.04086745991116762, 0.03861704258400202, 0.036724792236924174, 0.0353489439085722, 0.034173405121445656, 0.033291582461416724, 0.03260209433323145, 0.03197603836637735, 0.03142624312233925, 0.03094419389796257, 0.03052780131715536, 0.029986698361516, 0.029662011717677118, 0.02930098083367944, 0.029009738273620606, 0.028741251201957464]

pred std history
[0.9342638821213841, 0.968207675479889, 0.9747434713363647, 0.9777280665817261, 0.9793482710609436, 0.9805232583618164, 0.9814466710205079, 0.9821930579872131, 0.9828533185691833, 0.9833062073059082, 0.9836388912315369, 0.9840125755653382, 0.9842857368659973, 0.9844799619865418, 0.9849061526908874, 0.9849655881652832, 0.9852385511398315, 0.9853833767242431, 0.9855111967849731, 0.9856063576354981]
